In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Exploratory Data Analysis: Solar Radiation Dataset\n",
    "## Understanding patterns and relationships in solar radiation data"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import pandas as pd\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "import sys\n",
    "from pathlib import Path\n",
    "sys.path.append('..')\n",
    "\n",
    "from src.data_preprocessing import SolarDataPreprocessor\n",
    "import config\n",
    "\n",
    "%matplotlib inline\n",
    "plt.style.use('seaborn-v0_8-darkgrid')\n",
    "sns.set_palette(\"husl\")\n",
    "\n",
    "# Create results directory for saving plots\n",
    "results_dir = Path('../results')\n",
    "results_dir.mkdir(exist_ok=True)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Load data\n",
    "print(\"=\"*50)\n",
    "print(\"LOADING DATA\")\n",
    "print(\"=\"*50)\n",
    "preprocessor = SolarDataPreprocessor(str(config.DATA_FILE))\n",
    "df = preprocessor.load_data()\n",
    "exploration = preprocessor.explore_data()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Engineer features\n",
    "print(\"\\n\" + \"=\"*50)\n",
    "print(\"FEATURE ENGINEERING\")\n",
    "print(\"=\"*50)\n",
    "df = preprocessor.engineer_features()\n",
    "print(f\"\\nNew shape after feature engineering: {df.shape}\")\n",
    "print(f\"\\nNew columns: {[col for col in df.columns if col not in preprocessor.df.columns]}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "## 📊 1. Radiation Distribution"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "fig, axes = plt.subplots(2, 2, figsize=(15, 10))\n",
    "fig.suptitle('Radiation Distribution Analysis', fontsize=16, y=1.02)\n",
    "\n",
    "# Histogram with KDE\n",
    "axes[0, 0].hist(df['Radiation'], bins=50, edgecolor='black', alpha=0.7, density=True)\n",
    "df['Radiation'].plot.kde(ax=axes[0, 0], color='red', linewidth=2)\n",
    "axes[0, 0].axvline(df['Radiation'].mean(), color='green', linestyle='--', linewidth=2, label=f'Mean: {df[\"Radiation\"].mean():.2f}')\n",
    "axes[0, 0].axvline(df['Radiation'].median(), color='orange', linestyle='--', linewidth=2, label=f'Median: {df[\"Radiation\"].median():.2f}')\n",
    "axes[0, 0].set_xlabel('Radiation (W/m²)')\n",
    "axes[0, 0].set_ylabel('Density')\n",
    "axes[0, 0].set_title('Radiation Distribution with KDE')\n",
    "axes[0, 0].legend()\n",
    "axes[0, 0].grid(True, alpha=0.3)\n",
    "\n",
    "# Box plot\n",
    "bp = axes[0, 1].boxplot(df['Radiation'], patch_artist=True)\n",
    "bp['boxes'][0].set_facecolor('lightblue')\n",
    "axes[0, 1].set_ylabel('Radiation (W/m²)')\n",
    "axes[0, 1].set_title('Radiation Box Plot')\n",
    "axes[0, 1].grid(True, alpha=0.3)\n",
    "\n",
    "# Time series (first 1000 points)\n",
    "axes[1, 0].plot(df['Radiation'].values[:1000], linewidth=0.5, color='blue')\n",
    "axes[1, 0].set_xlabel('Time Step')\n",
    "axes[1, 0].set_ylabel('Radiation')\n",
    "axes[1, 0].set_title('Radiation Time Series (first 1000 points)')\n",
    "axes[1, 0].grid(True, alpha=0.3)\n",
    "\n",
    "# Q-Q plot\n",
    "from scipy import stats\n",
    "stats.probplot(df['Radiation'].values, dist=\"norm\", plot=axes[1, 1])\n",
    "axes[1, 1].set_title('Q-Q Plot')\n",
    "axes[1, 1].grid(True, alpha=0.3)\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.savefig(results_dir / 'radiation_distribution.png', dpi=150, bbox_inches='tight')\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "## ⏰ 2. Temporal Patterns"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "fig, axes = plt.subplots(2, 2, figsize=(15, 10))\n",
    "fig.suptitle('Temporal Patterns in Solar Radiation', fontsize=16, y=1.02)\n",
    "\n",
    "# Hourly pattern with confidence interval\n",
    "hourly_stats = df.groupby('Hour')['Radiation'].agg(['mean', 'std', 'count'])\n",
    "axes[0, 0].plot(hourly_stats.index, hourly_stats['mean'], marker='o', linewidth=2, color='blue')\n",
    "axes[0, 0].fill_between(hourly_stats.index, \n",
    "                        hourly_stats['mean'] - hourly_stats['std'],\n",
    "                        hourly_stats['mean'] + hourly_stats['std'],\n",
    "                        alpha=0.2, color='blue')\n",
    "axes[0, 0].set_xlabel('Hour of Day')\n",
    "axes[0, 0].set_ylabel('Average Radiation (W/m²)')\n",
    "axes[0, 0].set_title('Average Radiation by Hour (with ±1σ)')\n",
    "axes[0, 0].grid(True, alpha=0.3)\n",
    "axes[0, 0].set_xticks(range(0, 24, 2))\n",
    "\n",
    "# Daily pattern (box plot by hour)\n",
    "df.boxplot(column='Radiation', by='Hour', ax=axes[0, 1])\n",
    "axes[0, 1].set_xlabel('Hour of Day')\n",
    "axes[0, 1].set_ylabel('Radiation (W/m²)')\n",
    "axes[0, 1].set_title('Radiation Distribution by Hour')\n",
    "axes[0, 1].set_xticks(range(0, 24, 2))\n",
    "\n",
    "# Radiation vs TimeSin (colored by hour)\n",
    "sc = axes[1, 0].scatter(df['TimeSin'], df['Radiation'], c=df['Hour'], \n",
    "                        cmap='viridis', alpha=0.3, s=5)\n",
    "axes[1, 0].set_xlabel('TimeSin')\n",
    "axes[1, 0].set_ylabel('Radiation')\n",
    "axes[1, 0].set_title('Radiation vs TimeSin (colored by hour)')\n",
    "plt.colorbar(sc, ax=axes[1, 0], label='Hour')\n",
    "\n",
    "# Radiation vs TimeCos (colored by hour)\n",
    "sc = axes[1, 1].scatter(df['TimeCos'], df['Radiation'], c=df['Hour'], \n",
    "                        cmap='viridis', alpha=0.3, s=5)\n",
    "axes[1, 1].set_xlabel('TimeCos')\n",
    "axes[1, 1].set_ylabel('Radiation')\n",
    "axes[1, 1].set_title('Radiation vs TimeCos (colored by hour)')\n",
    "plt.colorbar(sc, ax=axes[1, 1], label='Hour')\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.savefig(results_dir / 'temporal_patterns.png', dpi=150, bbox_inches='tight')\n",
    "plt.show()\n",
    "\n",
    "# Print peak hour information\n",
    "peak_hour = hourly_stats['mean'].idxmax()\n",
    "print(f\"\\nPeak radiation hour: {peak_hour}:00\")\n",
    "print(f\"Peak average radiation: {hourly_stats['mean'].max():.2f} W/m²\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "## 🌡️ 3. Weather Feature Relationships"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "fig, axes = plt.subplots(2, 3, figsize=(18, 10))\n",
    "fig.suptitle('Weather Feature Relationships with Radiation', fontsize=16, y=1.02)\n",
    "\n",
    "weather_features = ['Temperature', 'Humidity', 'Pressure', 'Speed', 'WindDirection(Degrees)']\n",
    "\n",
    "for i, feature in enumerate(weather_features):\n",
    "    row = i // 3\n",
    "    col = i % 3\n",
    "    \n",
    "    # Hexbin plot for density\n",
    "    hb = axes[row, col].hexbin(df[feature], df['Radiation'], gridsize=30, cmap='YlOrRd')\n",
    "    axes[row, col].set_xlabel(feature)\n",
    "    axes[row, col].set_ylabel('Radiation')\n",
    "    axes[row, col].set_title(f'Radiation vs {feature}')\n",
    "    plt.colorbar(hb, ax=axes[row, col], label='Count')\n",
    "    \n",
    "    # Add trend line\n",
    "    z = np.polyfit(df[feature].values, df['Radiation'].values, 1)\n",
    "    p = np.poly1d(z)\n",
    "    x_trend = np.linspace(df[feature].min(), df[feature].max(), 100)\n",
    "    axes[row, col].plot(x_trend, p(x_trend), 'r-', linewidth=2, label=f'Slope: {z[0]:.2f}')\n",
    "    axes[row, col].legend()\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.savefig(results_dir / 'weather_relationships.png', dpi=150, bbox_inches='tight')\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "## 🔗 4. Correlation Analysis"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Select numeric columns for correlation\n",
    "numeric_cols = df.select_dtypes(include=[np.number]).columns\n",
    "corr_matrix = df[numeric_cols].corr()\n",
    "\n",
    "plt.figure(figsize=(16, 12))\n",
    "\n",
    "# Create mask for upper triangle\n",
    "mask = np.triu(np.ones_like(corr_matrix, dtype=bool))\n",
    "\n",
    "sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='coolwarm', center=0,\n",
    "            square=True, linewidths=0.5, cbar_kws={'label': 'Correlation'})\n",
    "plt.title('Feature Correlation Matrix', fontsize=16)\n",
    "plt.tight_layout()\n",
    "plt.savefig(results_dir / 'correlation_matrix.png', dpi=150, bbox_inches='tight')\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Top correlations with Radiation\n",
    "radiation_corr = corr_matrix['Radiation'].sort_values(ascending=False)\n",
    "\n",
    "print(\"\\n\" + \"=\"*50)\n",
    "print(\"TOP CORRELATIONS WITH RADIATION\")\n",
    "print(\"=\"*50)\n",
    "\n",
    "positive_corr = radiation_corr[radiation_corr > 0].sort_values(ascending=False)\n",
    "negative_corr = radiation_corr[radiation_corr < 0].sort_values()\n",
    "\n",
    "print(\"\\n📈 Positive Correlations:\")\n",
    "for feature, corr in positive_corr.head(10).items():\n",
    "    if feature != 'Radiation':\n",
    "        print(f\"   {feature:25s}: {corr:.4f}\")\n",
    "\n",
    "print(\"\\n📉 Negative Correlations:\")\n",
    "for feature, corr in negative_corr.head(10).items():\n",
    "    print(f\"   {feature:25s}: {corr:.4f}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "## ☀️ 5. Day/Night Analysis"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "fig, axes = plt.subplots(2, 2, figsize=(15, 10))\n",
    "fig.suptitle('Day/Night Analysis', fontsize=16, y=1.02)\n",
    "\n",
    "# Daytime vs Nighttime box plot\n",
    "daytime_rad = df[df['IsDaytime'] == 1]['Radiation']\n",
    "nighttime_rad = df[df['IsDaytime'] == 0]['Radiation']\n",
    "\n",
    "bp = axes[0, 0].boxplot([daytime_rad, nighttime_rad], labels=['Daytime', 'Nighttime'],\n",
    "                         patch_artist=True)\n",
    "bp['boxes'][0].set_facecolor('yellow')\n",
    "bp['boxes'][1].set_facecolor('darkblue')\n",
    "axes[0, 0].set_ylabel('Radiation (W/m²)')\n",
    "axes[0, 0].set_title('Radiation: Daytime vs Nighttime')\n",
    "axes[0, 0].grid(True, alpha=0.3)\n",
    "\n",
    "# Statistics\n",
    "stats_text = f\"Daytime:\\nMean: {daytime_rad.mean():.1f}\\nStd: {daytime_rad.std():.1f}\\n\\nNighttime:\\nMean: {nighttime_rad.mean():.1f}\\nStd: {nighttime_rad.std():.1f}\"\n",
    "axes[0, 0].text(1.1, 0.5, stats_text, transform=axes[0, 0].transAxes,\n",
    "                verticalalignment='center', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))\n",
    "\n",
    "# Distribution comparison\n",
    "axes[0, 1].hist(daytime_rad, bins=50, alpha=0.5, label='Daytime', density=True)\n",
    "axes[0, 1].hist(nighttime_rad, bins=50, alpha=0.5, label='Nighttime', density=True)\n",
    "axes[0, 1].set_xlabel('Radiation (W/m²)')\n",
    "axes[0, 1].set_ylabel('Density')\n",
    "axes[0, 1].set_title('Radiation Distribution: Day vs Night')\n",
    "axes[0, 1].legend()\n",
    "axes[0, 1].grid(True, alpha=0.3)\n",
    "\n",
    "# Time since sunrise effect\n",
    "df_sorted = df.sort_values('TimeSinceSunrise')\n",
    "axes[1, 0].scatter(df_sorted['TimeSinceSunrise'], df_sorted['Radiation'], \n",
    "                   alpha=0.1, s=5, c=df_sorted['IsDaytime'], cmap='coolwarm')\n",
    "axes[1, 0].axvline(x=0, color='r', linestyle='--', linewidth=2, label='Sunrise')\n",
    "axes[1, 0].axvline(x=df_sorted['DaylightMinutes'].iloc[0], color='orange', \n",
    "                   linestyle='--', linewidth=2, label='Sunset')\n",
    "axes[1, 0].set_xlabel('Minutes Since Sunrise')\n",
    "axes[1, 0].set_ylabel('Radiation')\n",
    "axes[1, 0].set_title('Radiation vs Time Since Sunrise')\n",
    "axes[1, 0].legend()\n",
    "axes[1, 0].grid(True, alpha=0.3)\n",
    "\n",
    "# Daylight hours distribution\n",
    "daylight_by_day = df.groupby(df['Data'].dt.date)['DaylightMinutes'].first()\n",
    "axes[1, 1].hist(daylight_by_day, bins=30, edgecolor='black', alpha=0.7, color='orange')\n",
    "axes[1, 1].axvline(daylight_by_day.mean(), color='red', linestyle='--', label=f'Mean: {daylight_by_day.mean()/60:.1f}h')\n",
    "axes[1, 1].set_xlabel('Daylight Minutes')\n",
    "axes[1, 1].set_ylabel('Frequency')\n",
    "axes[1, 1].set_title('Distribution of Daylight Hours')\n",
    "axes[1, 1].legend()\n",
    "axes[1, 1].grid(True, alpha=0.3)\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.savefig(results_dir / 'day_night_analysis.png', dpi=150, bbox_inches='tight')\n",
    "plt.show()\n",
    "\n",
    "print(f\"\\nDaytime mean radiation: {daytime_rad.mean():.2f} W/m²\")\n",
    "print(f\"Nighttime mean radiation: {nighttime_rad.mean():.2f} W/m²\")\n",
    "print(f\"Day/Night ratio: {daytime_rad.mean()/nighttime_rad.mean():.2f}x\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "## 📈 6. Time Series Decomposition"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from statsmodels.tsa.seasonal import seasonal_decompose\n",
    "from statsmodels.graphics.tsaplots import plot_acf, plot_pacf\n",
    "\n",
    "# Sample a continuous segment for decomposition\n",
    "segment_length = 2000\n",
    "segment = df['Radiation'].values[:segment_length]\n",
    "\n",
    "fig, axes = plt.subplots(4, 1, figsize=(15, 12))\n",
    "fig.suptitle('Time Series Decomposition (24h Seasonality)', fontsize=16, y=1.02)\n",
    "\n",
    "try:\n",
    "    # Perform seasonal decomposition (assuming daily seasonality = 24 hours)\n",
    "    decomposition = seasonal_decompose(segment, model='additive', period=24)\n",
    "    \n",
    "    axes[0].plot(decomposition.observed)\n",
    "    axes[0].set_title('Original Time Series')\n",
    "    axes[0].set_ylabel('Radiation')\n",
    "    axes[0].grid(True, alpha=0.3)\n",
    "    \n",
    "    axes[1].plot(decomposition.trend)\n",
    "    axes[1].set_title('Trend Component')\n",
    "    axes[1].set_ylabel('Radiation')\n",
    "    axes[1].grid(True, alpha=0.3)\n",
    "    \n",
    "    axes[2].plot(decomposition.seasonal[:100])  # Show first 100 points for clarity\n",
    "    axes[2].set_title('Seasonal Component (first 100 points)')\n",
    "    axes[2].set_ylabel('Radiation')\n",
    "    axes[2].grid(True, alpha=0.3)\n",
    "    \n",
    "    axes[3].plot(decomposition.resid)\n",
    "    axes[3].set_title('Residual Component')\n",
    "    axes[3].set_ylabel('Radiation')\n",
    "    axes[3].set_xlabel('Time Step')\n",
    "    axes[3].grid(True, alpha=0.3)\n",
    "    \n",
    "    plt.tight_layout()\n",
    "    plt.savefig(results_dir / 'time_series_decomposition.png', dpi=150, bbox_inches='tight')\n",
    "    plt.show()\n",
    "    \n",
    "except Exception as e:\n",
    "    print(f\"Decomposition error: {e}\")\n",
    "    print(\"\\nUsing autocorrelation analysis instead...\")\n",
    "    \n",
    "    # Autocorrelation plots\n",
    "    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))\n",
    "    plot_acf(segment, lags=48, ax=ax1)\n",
    "    ax1.set_title('Autocorrelation Function')\n",
    "    ax1.grid(True, alpha=0.3)\n",
    "    \n",
    "    plot_pacf(segment, lags=48, ax=ax2)\n",
    "    ax2.set_title('Partial Autocorrelation Function')\n",
    "    ax2.grid(True, alpha=0.3)\n",
    "    \n",
    "    plt.tight_layout()\n",
    "    plt.savefig(results_dir / 'autocorrelation.png', dpi=150, bbox_inches='tight')\n",
    "    plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "---\n",
    "## 🎯 7. Key Insights and Conclusions"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print(\"=\"*60)\n",
    "print(\"🔑 KEY INSIGHTS SUMMARY\")\n",
    "print(\"=\"*60)\n",
    "\n",
    "print(\"\\n📊 DATA OVERVIEW:\")\n",
    "print(f\"   - Total samples: {len(df):,}\")\n",
    "print(f\"   - Time span: {df['Data'].min()} to {df['Data'].max()}\")\n",
    "print(f\"   - Features after engineering: {len(df.columns)}\")\n",
    "\n",
    "print(\"\\n☀️ RADIATION STATISTICS:\")\n",
    "print(f\"   - Range: {df['Radiation'].min():.1f} - {df['Radiation'].max():.1f} W/m²\")\n",
    "print(f\"   - Mean: {df['Radiation'].mean():.1f} W/m²\")\n",
    "print(f\"   - Median: {df['Radiation'].median():.1f} W/m²\")\n",
    "print(f\"   - Std Dev: {df['Radiation'].std():.1f} W/m²\")\n",
    "print(f\"   - Zero values: {(df['Radiation'] == 0).sum()} ({(df['Radiation'] == 0).mean()*100:.2f}%)\")\n",
    "\n",
    "print(\"\\n⏰ TEMPORAL PATTERNS:\")\n",
    "peak_hour = hourly_stats['mean'].idxmax()\n",
    "print(f\"   - Peak radiation hour: {peak_hour}:00\")\n",
    "print(f\"   - Peak average radiation: {hourly_stats['mean'].max():.1f} W/m²\")\n",
    "print(f\"   - Lowest radiation hour: {hourly_stats['mean'].idxmin()}:00\")\n",
    "print(f\"   - Lowest average radiation: {hourly_stats['mean'].min():.1f} W/m²\")\n",
    "\n",
    "print(\"\\n🌡️ WEATHER CORRELATIONS:\")\n",
    "print(\"   Top 5 positive correlations:\")\n",
    "for feature, corr in positive_corr.head(5).items():\n",
    "    if feature != 'Radiation':\n",
    "        print(f\"      • {feature:20s}: {corr:.4f}\")\n",
    "\n",
    "print(\"\\n   Top 5 negative correlations:\")\n",
    "for feature, corr in negative_corr.head(5).items():\n",
    "    print(f\"      • {feature:20s}: {corr:.4f}\")\n",
    "\n",
    "print(\"\\n☀️ DAY/NIGHT ANALYSIS:\")\n",
    "print(f\"   - Daytime mean: {daytime_rad.mean():.1f} W/m²\")\n",
    "print(f\"   - Nighttime mean: {nighttime_rad.mean():.1f} W/m²\")\n",
    "print(f\"   - Day/Night ratio: {daytime_rad.mean()/nighttime_rad.mean():.1f}x\")\n",
    "print(f\"   - Average daylight hours: {df['DaylightMinutes'].mean()/60:.1f} hours\")\n",
    "\n",
    "print(\"\\n💡 RECOMMENDATIONS FOR MODELING:\")\n",
    "print(\"   1. Use LSTM/GRU to capture temporal dependencies (24h cycle)\")\n",
    "print(\"   2. Include cyclical time features (sin/cos of hour)\")\n",
    "print(\"   3. Consider day/night flag and time since sunrise\")\n",
    "print(\"   4. Temperature and humidity are key predictors\")\n",
    "print(\"   5. Handle zero-inflated nighttime data separately or use specialized loss\")\n",
    "print(\"   6. Use window size of 24 to capture daily patterns\")\n",
    "print(\"   7. Consider ensemble of models for different times of day\")\n",
    "print(\"   8. Add interaction features between temperature and humidity\")\n",
    "print(\"   9. Use attention mechanism to focus on relevant time steps\")\n",
    "print(\"   10. Validate model performance separately for day and night\")\n",
    "\n",
    "print(\"\\n\" + \"=\"*60)\n",
    "print(\"✅ EDA COMPLETE\")\n",
    "print(\"=\"*60)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Save processed data for modeling\n",
    "print(\"\\nSaving processed data for modeling...\")\n",
    "df.to_csv('../data/processed_solar_data.csv', index=False)\n",
    "print(\"✅ Data saved to '../data/processed_solar_data.csv'\")"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.9.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}